In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from torch.nn import functional as F
import pandas as pd
import matplotlib.pyplot as plt
import math

In [2]:
class AutoencoderDataset(Dataset):
    def __init__(self, image_dir):
        self.image_dir = image_dir
        self.images = sorted(os.listdir(image_dir))
        
        # 基本轉換
        self.transform = transforms.Compose([
            transforms.ToTensor(),
        ])
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.images[idx])
        
        # 讀取圖像
        image = Image.open(img_path).convert('RGB')
        
        # 調整圖像大小到統一尺寸
        target_size = (512, 512)
        image = image.resize(target_size, Image.BILINEAR)
        
        # 轉換
        image_tensor = self.transform(image)
        
        return image_tensor, image_tensor
class UNetAutoencoder(nn.Module):
    def __init__(self):
        super(UNetAutoencoder, self).__init__()
        
        # Encoder
        self.enc1 = self._double_conv(3, 64)      # 512x512x3  -> 512x512x64
        self.pool1 = nn.MaxPool2d(2, 2)           # 512x512x64 -> 256x256x64
        
        self.enc2 = self._double_conv(64, 128)    # 256x256x64 -> 256x256x128
        self.pool2 = nn.MaxPool2d(2, 2)           # 256x256x128 -> 128x128x128
        
        self.enc3 = self._double_conv(128, 256)   # 128x128x128 -> 128x128x256
        self.pool3 = nn.MaxPool2d(2, 2)           # 128x128x256 -> 64x64x256
        
        self.enc4 = self._double_conv(256, 512)   # 64x64x256 -> 64x64x512
        self.pool4 = nn.MaxPool2d(2, 2)           # 64x64x512 -> 32x32x512
        
        # Bottleneck
        self.bottleneck = self._double_conv(512, 1024)  # 32x32x512 -> 32x32x1024
        
        # Decoder
        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)  # 32x32x1024 -> 64x64x512
        self.dec4 = self._double_conv(1024, 512)  # 64x64x1024 -> 64x64x512 (包含skip connection)
        
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)   # 64x64x512 -> 128x128x256
        self.dec3 = self._double_conv(512, 256)   # 128x128x512 -> 128x128x256
        
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)   # 128x128x256 -> 256x256x128
        self.dec2 = self._double_conv(256, 128)   # 256x256x256 -> 256x256x128
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)    # 256x256x128 -> 512x512x64
        self.dec1 = self._double_conv(128, 64)    # 512x512x128 -> 512x512x64
        
        self.final_conv = nn.Conv2d(64, 3, kernel_size=1)  # 512x512x64 -> 512x512x3
        
    def _double_conv(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder path
        e1 = self.enc1(x)           # 512x512x64
        e2 = self.enc2(self.pool1(e1))   # 256x256x128
        e3 = self.enc3(self.pool2(e2))   # 128x128x256
        e4 = self.enc4(self.pool3(e3))   # 64x64x512
        
        # Bottleneck
        b = self.bottleneck(self.pool4(e4))   # 32x32x1024
        
        # Decoder path
        d4 = self.up4(b)           # 64x64x512
        d4 = torch.cat([d4, e4], dim=1)  # 64x64x1024
        d4 = self.dec4(d4)         # 64x64x512
        
        d3 = self.up3(d4)          # 128x128x256
        d3 = torch.cat([d3, e3], dim=1)  # 128x128x512
        d3 = self.dec3(d3)         # 128x128x256
        
        d2 = self.up2(d3)          # 256x256x128
        d2 = torch.cat([d2, e2], dim=1)  # 256x256x256
        d2 = self.dec2(d2)         # 256x256x128
        
        d1 = self.up1(d2)          # 512x512x64
        d1 = torch.cat([d1, e1], dim=1)  # 512x512x128
        d1 = self.dec1(d1)         # 512x512x64
        
        out = self.final_conv(d1)   # 512x512x3
        return torch.sigmoid(out)



def calculate_psnr(pred, target):
    """計算PSNR值"""
    mse = torch.mean((pred - target) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    psnr = 20 * torch.log10(max_pixel / torch.sqrt(mse))
    return psnr.item()

def plot_loss_and_psnr(losses, psnrs, save_path='training_curves.png'):
    """繪製損失和PSNR曲線"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.plot(losses, label='Training Loss')
    ax1.set_title('Training Loss Over Time')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(psnrs, label='PSNR')
    ax2.set_title('PSNR Over Time')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('PSNR (dB)')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def train_model(model, train_loader, criterion, optimizer, device, num_epochs=50):
    """訓練模型並記錄損失和PSNR"""
    losses = []
    psnrs = []
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        epoch_psnr = 0
        batch_count = 0
        
        for i, (images, targets) in enumerate(train_loader):
            images = images.to(device)
            targets = targets.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            # 計算PSNR
            psnr = calculate_psnr(outputs.detach(), targets)
            
            epoch_loss += loss.item()
            epoch_psnr += psnr
            batch_count += 1
            
            print(f'Epoch {epoch+1}/{num_epochs}, Batch {i}/{len(train_loader)-1}, '
                  f'Loss: {loss.item():.4f}, PSNR: {psnr:.2f}dB')
        
        avg_loss = epoch_loss / batch_count
        avg_psnr = epoch_psnr / batch_count
        losses.append(avg_loss)
        psnrs.append(avg_psnr)
        
        print(f'Epoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}, '
              f'Average PSNR: {avg_psnr:.2f}dB')
        
        # 保存模型
        if (epoch + 1) % 10 == 0:
            torch.save(model.state_dict(), f'3_2_autoencoder_model_epoch_{epoch+1}.pth')
    
    # 繪製訓練曲線
    plot_loss_and_psnr(losses, psnrs, save_path='3-2. loss curve.png')
    
    return losses, psnrs

def visualize_results(original, reconstruction, image_name, save_path):
    """生成原圖和重建圖的對比圖"""
    plt.figure(figsize=(10, 5))
    
    # 設置字體大小
    plt.rcParams.update({'font.size': 10})
    
    # 原始圖像
    plt.subplot(121)
    plt.title(f'Original\n{image_name}', pad=10)
    plt.imshow(original.permute(1, 2, 0))
    plt.axis('off')
    
    # 重建圖像
    plt.subplot(122)
    plt.title('Reconstruction', pad=10)
    plt.imshow(reconstruction.permute(1, 2, 0))
    plt.axis('off')
    
    plt.tight_layout(pad=3.0)
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close()

def test_model(model, test_dir, results_dir, device):
    """測試模型並生成結果"""
    model.eval()
    
    # 創建結果目錄
    os.makedirs(results_dir, exist_ok=True)
    results = []
    
    # 設置轉換
    transform = transforms.Compose([
        transforms.ToTensor(),
    ])
    
    # 處理測試集圖像
    for i in range(1, 20):  # 假設測試集有19張圖片
        img_name = f"{i:02d}_real_A.png"
        img_path = os.path.join(test_dir, img_name)
        
        if not os.path.exists(img_path):
            print(f"Warning: Missing file {img_name}")
            continue
        
        # 讀取原始圖像
        original = Image.open(img_path).convert('RGB')
        original_size = original.size
        
        # 處理輸入圖像
        input_tensor = transform(original)
        input_tensor = input_tensor.unsqueeze(0).to(device)
        input_tensor = F.interpolate(input_tensor, size=(512, 512), mode='bilinear', align_corners=True)
        
        # 重建
        with torch.no_grad():
            output = model(input_tensor)
        
        # 將輸出轉換回原始大小
        output = F.interpolate(output, size=original_size, mode='bilinear', align_corners=True)
        
        # 計算PSNR
        psnr = calculate_psnr(output.squeeze(), input_tensor.squeeze())
        
        # 保存結果
        results.append({
            'Image': f'Image_{i:02d}',
            'PSNR': psnr
        })
        
        # 生成視覺化結果
        save_path = os.path.join(results_dir, f'reconstruction_{i:02d}.png')
        visualize_results(
            input_tensor.squeeze().cpu(),
            output.squeeze().cpu(),
            f'Image {i:02d}',
            save_path
        )
    
    # 創建DataFrame並保存為CSV
    df = pd.DataFrame(results)
    
    # 添加平均值行
    avg_row = {
        'Image': 'Average',
        'PSNR': df['PSNR'].mean()
    }
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
    
    csv_path = os.path.join(results_dir, 'psnr_metrics.csv')
    df.to_csv(csv_path, index=False)
    
    # 打印結果摘要
    print("\nTest Results Summary:")
    print(df.to_string(index=False))
    
    return df



In [3]:
# 設置設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


# 創建數據加載器
train_dataset = AutoencoderDataset(
    image_dir='DRIVE/training/images'
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"Total number of training samples: {len(train_dataset)}")
print(f"Number of batches per epoch: {len(train_loader)}")

# 初始化模型
model = UNetAutoencoder().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 訓練模型
losses, psnrs = train_model(model, train_loader, criterion, optimizer, device, num_epochs=50)

# 測試模型
results_df = test_model(
    model,
    test_dir='DIRVE_TestingSet',
    results_dir='3-2_autoencoder_results',
    device=device
)


Using device: cuda
Total number of training samples: 20
Number of batches per epoch: 5
Epoch 1/50, Batch 0/4, Loss: 0.1136, PSNR: 9.45dB
Epoch 1/50, Batch 1/4, Loss: 0.0782, PSNR: 11.07dB
Epoch 1/50, Batch 2/4, Loss: 0.0403, PSNR: 13.95dB
Epoch 1/50, Batch 3/4, Loss: 0.0460, PSNR: 13.38dB
Epoch 1/50, Batch 4/4, Loss: 0.0430, PSNR: 13.66dB
Epoch 1/50, Average Loss: 0.0642, Average PSNR: 12.30dB
Epoch 2/50, Batch 0/4, Loss: 0.0291, PSNR: 15.37dB
Epoch 2/50, Batch 1/4, Loss: 0.0305, PSNR: 15.15dB
Epoch 2/50, Batch 2/4, Loss: 0.0184, PSNR: 17.36dB
Epoch 2/50, Batch 3/4, Loss: 0.0248, PSNR: 16.05dB
Epoch 2/50, Batch 4/4, Loss: 0.0188, PSNR: 17.25dB
Epoch 2/50, Average Loss: 0.0243, Average PSNR: 16.23dB
Epoch 3/50, Batch 0/4, Loss: 0.0114, PSNR: 19.42dB
Epoch 3/50, Batch 1/4, Loss: 0.0162, PSNR: 17.90dB
Epoch 3/50, Batch 2/4, Loss: 0.0167, PSNR: 17.78dB
Epoch 3/50, Batch 3/4, Loss: 0.0239, PSNR: 16.21dB
Epoch 3/50, Batch 4/4, Loss: 0.0189, PSNR: 17.22dB
Epoch 3/50, Average Loss: 0.0174, Ave